# Table fLUC IPCC regions

In [ ]:
# Libraries
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd

In [ ]:
# Directories
dir_zen = '../paper_ecosystem_carbon/output/zenodo/'
dir08 = '../paper_ecosystem_carbon/output/08_eval/'
dir_iturbide = '../data/iturbide2020/'

---

### Pre-processing

In [ ]:
# Get ipcc region shapefile data
geodf_ipcc = gpd.read_file(f'{dir_iturbide}'
                           'IPCC-WGI-reference-regions-v4_shapefile/'
                           'IPCC-WGI-reference-regions-v4.shp')

# Remove water regions and and antarctica
geodf_ipcc = geodf_ipcc[(geodf_ipcc['Type'].isin(['Land', 'Land-Ocean'])) & 
                        (~geodf_ipcc['Acronym'].isin(['EAN', 'WAN']))]

# List with ipcc region acronyms
list_acr = list(geodf_ipcc['Acronym'])

In [ ]:
# Get cspatial carbon density data
ds_in = xr.open_dataset(f'{dir_zen}ecosystem_carbon_data_figure_m05.nc') 

# Split and sum data
def get_ds_sub(s):
    return (
        ds_in[
            [
                var for var in ds_in.data_vars
                if var.startswith(f'{s}_') and 'pft' not in var
            ]
        ]
        .rename({
            var: var.removeprefix(f'{s}_fluc_')
            for var in ds_in.data_vars
            if var.startswith(f'{s}_') and 'pft' not in var
        })
    )

ds_chansis = get_ds_sub('chansis')
ds_cspatial = get_ds_sub('cspatial')
ds_cdgvm = get_ds_sub('cdgvm')

---

### Prepare dataframe with fluc estimates for each IPCC region

In [ ]:
# Prepare dataframe with regional fluc for each blue simulation
def prep_df_sum(ds_sel, year0=1960, year1=2023):

    """Calculates cummulative LUC fluxes for global and IPCC regions"""
    
    # list of regions
    list_acr = ['global', *list(geodf_ipcc['Acronym'])]
    
    # Prepare df_sum columns
    df_sum_mapping = {
        'reg': str,
        'name': str,
        'net_pgc': float,
        'deforestation_pgc': float,
        'forest_regrowth_pgc': float,
        'wood_harvest_pgc': float,
        'other_transitions_pgc': float,
    }
    
    # Initialize dataframe for eluc sums
    df_sum = pd.DataFrame(columns=df_sum_mapping.keys()).astype(df_sum_mapping)
    
    # Iterate  through list_acr
    for acr in list_acr:
        # Calculate global carbon totals per grid cell
        if acr == 'global':
            ds_sel_sum = (ds_sel * ds_in.cell_area)
        # Clip to region and calculate carbon per grid cell
        if acr != 'global':
            geodf_sel = geodf_ipcc[geodf_ipcc['Acronym'] == acr]
            
            ds_sel_sum = (ds_sel * ds_in.cell_area) \
                .rio.write_crs("EPSG:4326", inplace=True) \
                .rio.set_spatial_dims(x_dim='lon', y_dim='lat') \
                .rio.clip(geodf_sel.geometry.values, geodf_sel.crs, all_touched=False)
    
        # Adjust variable names to match defined column names
        ds_sel_sum = ds_sel_sum.rename(
            {i: i[5:] for i in list(ds_sel.data_vars) if i.startswith('eluc')})
        # Drop non relevant columns
        # Select time period, 
        # Calculate cummulative carbon
        # Transform to dataframe

        df_sel = (ds_sel_sum \
                .sel(time=slice(year0, year1)) \
                .sum(['lat', 'lon']) * 0.000000001) \
                .compute().sum('time').to_pandas().to_frame().T
        df_sel.columns = [f'{i}_pgc' for i in df_sel.columns]
        df_sel['reg'] = acr
        if acr == 'global':
            df_sel['name'] = 'Global'
        else:
            df_sel['name'] = geodf_ipcc.loc[geodf_ipcc['Acronym'] == acr, 'Name'].item()
        df_sel = df_sel[df_sum_mapping.keys()]
        # Add as row to df_sum
        df_sum = pd.concat([df_sum, df_sel], ignore_index=True)
    
    # Caluclate net share
    v_global = df_sum.loc[df_sum.reg == 'global', 'net_pgc'].item()
    df_sum = df_sum.assign(net_share_global = (df_sum['net_pgc'] / v_global) * 100)
    
    # Reorder columns
    df_sum = df_sum[['reg', 'name', 'net_share_global', 'net_pgc', 
                     'deforestation_pgc', 'forest_regrowth_pgc',
                     'wood_harvest_pgc', 'other_transitions_pgc']]
    
    return df_sum


df_sum_chansis = prep_df_sum(ds_chansis)
df_sum_cspatial = prep_df_sum(ds_cspatial)
df_sum_cdgvm = prep_df_sum(ds_cdgvm)

In [ ]:
# Prepare dataframe with regional differences betwenn chansis and cspatial 
# and between chansis and cdgvm

def prep_df_diff(df_in0, df_in1):

    """Calculates differences in cummulative LUC fluxes for global and IPCC 
    regions between blue simulations"""
    
    # Prepare dataframes
    df0 = df_in0.set_index('reg').drop(columns=['name', 'net_share_global']) 
    df1 = df_in1.set_index('reg').drop(columns=['name', 'net_share_global'])
    
    # Calculate difference in pgc
    df_diff_pgc = (df1 - df0)
    df_diff_pgc.columns = [f'{i[:-4]}_diff_pgc' for i in df_diff_pgc.columns]
    
    # Calculate difference in percent
    df_diff_per = (((df1 / df0)) - 1) * 100
    df_diff_per.columns = [f'{i[:-4]}_diff_per' for i in df_diff_per.columns]
    
    # Concat df0 and differences in pgc and percent
    df_diff = df_in0[['reg', 'name']] \
        .merge(df_diff_pgc, on='reg') \
        .merge(df_diff_per, on='reg')
    
    return df_diff


df_diff_cspatial = prep_df_diff(df_sum_chansis, df_sum_cspatial)
df_diff_cdgvm = prep_df_diff(df_sum_chansis, df_sum_cdgvm)

In [ ]:
# Create table with selected columns for supplementary materials
df_sum_chansis_sel = df_sum_chansis \
    [['reg', 'name', 'net_share_global', 'net_pgc']] \
    .rename(columns={'net_share_global': 'chansis_net_share_global', 
                     'net_pgc': 'chansis_net_pgc'})

df_sum_cspatial_sel = df_sum_cspatial[['reg', 'net_pgc']] \
    .rename(columns={'net_pgc': 'cspatial_net_pgc'})

df_sum_cdgvm_sel = df_sum_cdgvm[['reg', 'net_pgc']] \
    .rename(columns={'net_pgc': 'cdgvm_net_pgc'})

df_diff_cspatial_sel = df_diff_cspatial[['reg', 'net_diff_pgc', 'net_diff_per']] \
    .rename(columns={'net_diff_pgc': 'cspatial_net_diff_pgc', 
                     'net_diff_per': 'cspatial_net_diff_per'})

df_diff_cdgvm_sel = df_diff_cdgvm[['reg', 'net_diff_pgc', 'net_diff_per']] \
    .rename(columns={'net_diff_pgc': 'cdgvm_net_diff_pgc', 
                     'net_diff_per': 'cdgvm_net_diff_per'})

df_merge_sel = df_sum_chansis_sel \
    .merge(df_sum_cspatial_sel, on='reg') \
    .merge(df_diff_cspatial_sel, on='reg') \
    .merge(df_sum_cdgvm_sel, on='reg') \
    .merge(df_diff_cdgvm_sel, on='reg')

df_merge_sel = df_merge_sel.round(1)

# Export
df_merge_sel.to_csv(f'{dir08}tables/table_ipcc_region_fluc_sel.csv', 
                    mode='w', index=False)

In [ ]:
df_merge_sel